SEQUENTIAL WORKFLOW

In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
model = ChatOpenAI()

In [11]:
# create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [12]:
# define node
def llm_qa(state: LLMState)->LLMState:

    # extract qns from state
    question = state['question']

    # form a prompt
    prompt = f'answer the following question {question}'

    # ask that qns to llm
    try:
        answer = model.invoke(prompt).content
    except Exception as exc:
        if "insufficient_quota" in str(exc).lower() or "ratelimiterror" in type(exc).__name__.lower():
            answer = "Unable to answer because the OpenAI quota has been exhausted."
        else:
            raise

    # update the ans to state
    state['answer'] = answer

    return state

In [13]:
# create a graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

# compile
workflow = graph.compile()

In [14]:
# execute
initial_state = {'question': 'what is the distance of sun from earth'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'question': 'what is the distance of sun from earth', 'answer': 'Unable to answer because the OpenAI quota has been exhausted.'}
